In [ ]:
#Data Assesment


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import pandas as pd
import glob

# Ambil semua file checkpoint dari folder processed
files = glob.glob("data/processed/checkpoint_*.csv")

# Merge semua jadi satu
df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

print(f"Total baris sebelum cleaning: {len(df)}")

In [ ]:
# Definisikan kategori standar (sesuaikan dengan project kalian)
STANDARD_ROLES = {
    "Data Scientist"     : ["data scientist", "ml scientist", "research scientist"],
    "Data Analyst"       : ["data analyst", "business analyst", "bi analyst", "power bi", "tableau"],
    "Data Engineer"      : ["data engineer", "etl", "data pipeline", "spark", "airflow"],
    "ML Engineer"        : ["machine learning engineer", "mlops", "ai engineer", "deep learning"],
    "Backend Developer"  : ["backend", "back end", "python developer", "java developer", "api developer"],
    "Frontend Developer" : ["frontend", "front end", "react", "vue", "angular", "ui developer"],
    "Fullstack Developer": ["fullstack", "full stack", "full-stack"],
    "DevOps Engineer"    : ["devops", "devsecops", "cloud engineer", "sre", "infrastructure"],
    "Mobile Developer"   : ["mobile", "android", "ios", "flutter", "react native"],
    "QA Engineer"        : ["qa", "quality assurance", "tester", "automation engineer"],
    "BI Developer"       : ["bi developer", "business intelligence", "power bi development", "looker"],
    "Software Engineer"  : ["software engineer", "programmer", "developer"],  # fallback umum
}

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

def standardize_role(title):
    title_lower = title.lower().strip()
    
    # Step 1: Keyword exact match dulu (cepat)
    for std_role, keywords in STANDARD_ROLES.items():
        if any(kw in title_lower for kw in keywords):
            return std_role
    
    # Step 2: Cosine similarity jika tidak ada match
    all_keywords = [" ".join(v) for v in STANDARD_ROLES.values()]
    role_names   = list(STANDARD_ROLES.keys())
    
    vectorizer = TfidfVectorizer()
    tfidf      = vectorizer.fit_transform(all_keywords + [title_lower])
    sims       = cosine_similarity(tfidf[-1], tfidf[:-1])[0]
    
    best_idx   = sims.argmax()
    best_score = sims[best_idx]
    
    # Kalau similarity terlalu rendah → tandai sebagai "Other"
    return role_names[best_idx] if best_score > 0.1 else "Other"

# Terapkan ke dataframe
df['standardized_role'] = df['title'].apply(standardize_role)

print(df['standardized_role'].value_counts())